In [ ]:
# Cell 1 — Install (restart kernel after this cell)
import subprocess
subprocess.run(['pip', 'uninstall', 'unsloth', 'unsloth_zoo', 'torchao', 'trl', 'transformers', 'bitsandbytes', '-y'])
subprocess.run(['pip', 'install', '-q',
    'transformers==4.46.3',
    'trl==0.12.2',
    'peft==0.13.2',
    'accelerate==1.1.1',
    'datasets',
    'seqeval',
    'tqdm',
    'faiss-cpu',
    'sentence-transformers',
])
print('Done. Restart kernel now, then run all cells below.')

In [1]:
# Cell 2 — Imports
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset, Dataset
from seqeval.metrics import f1_score, classification_report
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

random.seed(42)
torch.manual_seed(42)
print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

2026-05-15 17:48:28.069479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778867308.093250     203 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778867308.103920     203 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778867308.131244     203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778867308.131264     203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778867308.131266     203 computation_placer.cc:177] computation placer alr

Imports OK
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Free GPU: 15.5 GB


In [2]:
# Cell 3 — Load dataset
ds = load_dataset('jjzha/sayfullina')
print('Train:', len(ds['train']))
print('Test: ', len(ds['test']))
print('Sample:', ds['train'][0])

Train: 3705
Test:  1851
Sample: {'idx': 0, 'tokens': ['excellent', 'organisational', 'skill', 'and', 'able', 'to', 'multitask'], 'tags_skill': ['O', 'O', 'O', 'O', 'B-SOFT', 'I-SOFT', 'I-SOFT']}


In [3]:
# Cell 4 — BIO helpers
def get_tokens_and_tags(example):
    return example['tokens'], example['tags_skill']


def to_bracket_format(tokens, tags):
    result = []
    i = 0
    while i < len(tokens):
        if tags[i].startswith('B-'):
            span = [tokens[i]]
            j = i + 1
            while j < len(tokens) and tags[j].startswith('I-'):
                span.append(tokens[j])
                j += 1
            result.append(f"@@{' '.join(span)}##")
            i = j
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)


def bio_from_bracket(tokens, output_text):
    tags = ['O'] * len(tokens)
    spans = []
    text = output_text
    while '@@' in text and '##' in text:
        start = text.find('@@')
        end   = text.find('##', start)
        if start == -1 or end == -1:
            break
        span = text[start+2:end].strip()
        spans.append(span)
        text = text[end+2:]
    for span in spans:
        span_tokens = span.split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i : i + n] == span_tokens:
                if all(tags[i + j] == 'O' for j in range(n)):
                    tags[i] = 'B-SOFT'
                    for j in range(1, n):
                        tags[i + j] = 'I-SOFT'
                break
    return tags


def spans_from_bio(tokens, tags):
    spans, current = [], []
    for token, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            if current:
                spans.append(' '.join(current))
            current = [token]
        elif tag.startswith('I-') and current:
            current.append(token)
        else:
            if current:
                spans.append(' '.join(current))
            current = []
    if current:
        spans.append(' '.join(current))
    return spans

print('BIO helpers defined')

BIO helpers defined


In [4]:
# Cell 5 — Dataset-specific ICL prompt (Sayfullina)
# This is the SAME prompt used at inference — baking it into SFT training
# is the key fix from Phase 5 (where prompt mismatch dropped F1 from 0.815 to 0.755)
SYSTEM_ICL = (
    'You are a soft-skill extractor specialising in job advertisement sentences. '
    'Your task is to identify ONLY soft skills — interpersonal, behavioural, and '
    'personal competencies such as communication, leadership, teamwork, adaptability. '
    'Do NOT tag hard skills, tools, technologies, qualifications, or job titles. '
    'Sentences are pre-tokenised and may be grammatically incomplete — extract spans '
    'exactly as they appear, token-by-token. '
    'Wrap each soft skill span with @@...## brackets. '
    'Copy the sentence exactly with no other changes. '
    'If there are no soft skills, return the sentence unchanged. '
    'Output only the annotated sentence — no explanation, no commentary.'
)


def format_prompt(sentence, target=None):
    prompt = (
        f'<|im_start|>system\n{SYSTEM_ICL}<|im_end|>\n'
        f'<|im_start|>user\n{sentence}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    if target is not None:
        prompt += f'{target}<|im_end|>'
    return prompt


sample_tokens, _ = get_tokens_and_tags(ds['train'][0])
print('Sample prompt:')
print(format_prompt(' '.join(sample_tokens)))

Sample prompt:
<|im_start|>system
You are a soft-skill extractor specialising in job advertisement sentences. Your task is to identify ONLY soft skills — interpersonal, behavioural, and personal competencies such as communication, leadership, teamwork, adaptability. Do NOT tag hard skills, tools, technologies, qualifications, or job titles. Sentences are pre-tokenised and may be grammatically incomplete — extract spans exactly as they appear, token-by-token. Wrap each soft skill span with @@...## brackets. Copy the sentence exactly with no other changes. If there are no soft skills, return the sentence unchanged. Output only the annotated sentence — no explanation, no commentary.<|im_end|>
<|im_start|>user
excellent organisational skill and able to multitask<|im_end|>
<|im_start|>assistant



In [5]:
import os
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if 'skill' in f.lower():
            print(os.path.join(root, f))


/kaggle/input/datasets/acme105/skills-en-esco/skills_en.csv


In [6]:
# Cell 6 — Build FAISS indexes (RAG-1: training sentences, RAG-2: ESCO skills)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# RAG-1: index all training sentences that have at least one skill
train_with_skills = [
    ex for ex in ds['train']
    if any(t != 'O' for t in ex['tags_skill'])
]
train_sentences = [' '.join(ex['tokens']) for ex in train_with_skills]

print(f'Embedding {len(train_sentences)} training sentences for RAG-1...')
rag1_emb = embedder.encode(train_sentences, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag1_emb)
rag1_index = faiss.IndexFlatIP(rag1_emb.shape[1])
rag1_index.add(rag1_emb)
print(f'RAG-1 index: {rag1_index.ntotal} entries')

# RAG-2: index ESCO skills
ESCO_PATH = '/kaggle/input/datasets/acme105/skills-en-esco/skills_en.csv'
esco = pd.read_csv(ESCO_PATH)
print(f'ESCO rows: {len(esco)}')

def build_esco_text(row):
    parts = [str(row.get('preferredLabel', ''))]
    alt = str(row.get('altLabels', ''))
    if alt and alt != 'nan':
        parts.append(alt.replace('\n', ', '))
    desc = str(row.get('description', ''))
    if desc and desc != 'nan':
        parts.append(desc[:200])
    return ' | '.join(parts)

esco_texts  = [build_esco_text(row) for _, row in esco.iterrows()]
esco_labels = esco['preferredLabel'].tolist()

print(f'Embedding {len(esco_texts)} ESCO skills for RAG-2...')
rag2_emb = embedder.encode(esco_texts, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag2_emb)
rag2_index = faiss.IndexFlatIP(rag2_emb.shape[1])
rag2_index.add(rag2_emb)
print(f'RAG-2 index: {rag2_index.ntotal} entries')

Embedding 3701 training sentences for RAG-1...


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

RAG-1 index: 3701 entries
ESCO rows: 13960
Embedding 13960 ESCO skills for RAG-2...


Batches:   0%|          | 0/219 [00:00<?, ?it/s]

RAG-2 index: 13960 entries


In [7]:
# Cell 7 — RAG retrieval functions
def retrieve_rag1(query_tokens, k=3):
    query = ' '.join(query_tokens)
    q_emb = embedder.encode([query]).astype('float32')
    faiss.normalize_L2(q_emb)
    _, indices = rag1_index.search(q_emb, k)
    results = []
    for idx in indices[0]:
        ex = train_with_skills[idx]
        tokens, tags = get_tokens_and_tags(ex)
        results.append((tokens, tags))
    return results


def retrieve_rag2(candidate_spans, k=3):
    if not candidate_spans:
        return ''
    definitions = []
    for span in candidate_spans[:5]:
        q_emb = embedder.encode([span]).astype('float32')
        faiss.normalize_L2(q_emb)
        _, indices = rag2_index.search(q_emb, k)
        for idx in indices[0]:
            label = esco_labels[idx]
            desc  = str(esco.iloc[idx].get('description', ''))
            if desc and desc != 'nan':
                definitions.append(f'- {label}: {desc[:120]}')
    seen, unique = set(), []
    for d in definitions:
        if d not in seen:
            seen.add(d)
            unique.append(d)
    return '\n'.join(unique[:8])


def build_augmented_sentence(query_tokens, rag1_examples, esco_context):
    """Combine RAG-1 few-shot examples + RAG-2 ESCO definitions into the user turn."""
    lines = []
    if rag1_examples:
        lines.append('Examples of soft-skill annotation:')
        for ex_tokens, ex_tags in rag1_examples:
            annotated = to_bracket_format(ex_tokens, ex_tags)
            lines.append(f'  Input:  {" ".join(ex_tokens)}')
            lines.append(f'  Output: {annotated}')
        lines.append('')
    if esco_context:
        lines.append('Relevant ESCO soft-skill definitions:')
        lines.append(esco_context)
        lines.append('')
    lines.append(' '.join(query_tokens))
    return '\n'.join(lines)

print('RAG retrieval functions defined')

RAG retrieval functions defined


In [8]:
# Cell 8 — Build SFT training set with RAG-augmented prompts
# For each training example: retrieve RAG-1 neighbours + RAG-2 ESCO context,
# then format with the ICL prompt. This is the core Phase 6 change.

def build_training_set_with_rag(dataset):
    positives, negatives = [], []
    for ex in tqdm(dataset['train'], desc='Building RAG-augmented training set'):
        tokens, tags = get_tokens_and_tags(ex)
        target = to_bracket_format(tokens, tags)

        # Get RAG-1 neighbours (exclude self by checking exact sentence match)
        rag1_examples = retrieve_rag1(tokens, k=4)
        self_sentence = ' '.join(tokens)
        rag1_examples = [
            (t, tg) for t, tg in rag1_examples
            if ' '.join(t) != self_sentence
        ][:3]

        # RAG-2: use predicted spans (from gold tags) as anchors for ESCO lookup
        gold_spans = spans_from_bio(tokens, tags)
        esco_context = retrieve_rag2(gold_spans if gold_spans else [' '.join(tokens)], k=2)

        user_text = build_augmented_sentence(tokens, rag1_examples, esco_context)
        entry = {'text': format_prompt(user_text, target)}

        if '@@' in target:
            positives.append(entry)
        else:
            negatives.append(entry)

    max_neg   = int(len(positives) * 0.3)
    negatives = random.sample(negatives, min(max_neg, len(negatives)))
    combined  = positives + negatives
    random.shuffle(combined)
    print(f'Positives: {len(positives)}, Hard negatives: {len(negatives)}, Total: {len(combined)}')
    return combined


train_data = build_training_set_with_rag(ds)
hf_train   = Dataset.from_list(train_data)
print('Sample prompt (truncated):')
print(train_data[0]['text'][:600])

Building RAG-augmented training set: 100%|██████████| 3705/3705 [00:51<00:00, 72.43it/s]


Positives: 3701, Hard negatives: 4, Total: 3705
Sample prompt (truncated):
<|im_start|>system
You are a soft-skill extractor specialising in job advertisement sentences. Your task is to identify ONLY soft skills — interpersonal, behavioural, and personal competencies such as communication, leadership, teamwork, adaptability. Do NOT tag hard skills, tools, technologies, qualifications, or job titles. Sentences are pre-tokenised and may be grammatically incomplete — extract spans exactly as they appear, token-by-token. Wrap each soft skill span with @@...## brackets. Copy the sentence exactly with no other changes. If there are no soft skills, return the sentence uncha


In [9]:
# Cell 9 — Load model + LoRA
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': 'cuda:0'},
    trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.enable_input_require_grads()
print(f'GPU memory after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Free GPU: 15.4 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
GPU memory after LoRA: 6.4 GB


In [10]:
# Cell 10 — Train
sft_config = SFTConfig(
    output_dir                    = '/kaggle/working/checkpoints',
    per_device_train_batch_size   = 2,
    gradient_accumulation_steps   = 8,
    num_train_epochs              = 3,
    learning_rate                 = 2e-4,
    warmup_ratio                  = 0.1,
    lr_scheduler_type             = 'cosine',
    fp16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    logging_steps                 = 20,
    save_strategy                 = 'epoch',
    report_to                     = 'none',
    seed                          = 42,
    optim                         = 'adamw_torch',
    ddp_find_unused_parameters    = False,
    dataloader_pin_memory         = False,
    dataset_text_field            = 'text',
    max_seq_length                = 512,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = hf_train,
    args          = sft_config,
)

trainer.train()
print('Training complete.')

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Step,Training Loss
20,3.165900
40,1.336500
60,0.927400
80,0.799100
100,0.722200
120,0.670400
140,0.623000
160,0.608200
180,0.582300
200,0.533900


Training complete.


In [11]:
# Cell 11 — Save adapter
model.save_pretrained('/kaggle/working/lora_adapter_p6')
tokenizer.save_pretrained('/kaggle/working/lora_adapter_p6')
print('Saved:', os.listdir('/kaggle/working/lora_adapter_p6'))

Saved: ['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'special_tokens_map.json', 'merges.txt', 'added_tokens.json', 'tokenizer_config.json', 'vocab.json']


In [12]:
# Cell 12 — BIO Verifier
def verify_bio(tags):
    prev_tag = 'O'
    for i, tag in enumerate(tags):
        if tag == 'O':
            prev_tag = 'O'
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I':
            if prev_tag == 'O':
                return False, f'I-tag at position {i} ({tag}) follows O.'
            _, prev_label = prev_tag.split('-', 1)
            if prev_label != label:
                return False, f'I-tag at position {i} ({tag}) follows {prev_tag} — type mismatch.'
        prev_tag = tag
    return True, ''


def fix_bio(tags):
    fixed = list(tags)
    prev_tag = 'O'
    for i, tag in enumerate(fixed):
        if tag == 'O':
            prev_tag = 'O'
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I':
            if prev_tag == 'O':
                fixed[i] = f'B-{label}'
            else:
                _, prev_label = prev_tag.split('-', 1)
                if prev_label != label:
                    fixed[i] = f'B-{label}'
        prev_tag = fixed[i]
    return fixed

print('Verifier defined')

Verifier defined


In [13]:
# Cell 13 — Inference with RAG + Verifier
model.eval()

MAX_RETRIES = 3

def predict_with_rag_and_verify(tokens, max_retries=MAX_RETRIES):
    """
    1. Retrieve RAG-1 neighbours + RAG-2 ESCO context
    2. Build augmented prompt
    3. Generate with model
    4. Parse BIO; if illegal, feed error back and regenerate (up to max_retries)
    5. Fallback: deterministic fix_bio if still illegal after all retries
    """
    rag1_examples = retrieve_rag1(tokens, k=3)
    esco_context  = retrieve_rag2([' '.join(tokens)], k=3)
    user_text     = build_augmented_sentence(tokens, rag1_examples, esco_context)

    # Build conversation history for iterative correction
    prompt = format_prompt(user_text)
    current_bio = ['O'] * len(tokens)

    for attempt in range(max_retries):
        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=512,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = 128,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )

        decoded = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        )

        current_bio = bio_from_bracket(tokens, decoded)
        valid, error = verify_bio(current_bio)

        if valid:
            return current_bio

        # Append correction feedback and retry
        correction = (
            f'{decoded}<|im_end|>\n'
            f'<|im_start|>user\n'
            f'BIO error: {error} '
            f'Please output the corrected annotated sentence only.<|im_end|>\n'
            f'<|im_start|>assistant\n'
        )
        prompt = prompt + correction

    # Deterministic fallback
    return fix_bio(current_bio)

print('Inference function defined')

Inference function defined


In [14]:
# Cell 14 — Sanity check: print 3 raw outputs
for i in range(3):
    tokens, gold = get_tokens_and_tags(ds['test'][i])
    pred = predict_with_rag_and_verify(tokens)
    print(f'Input: {" ".join(tokens)}')
    print(f'Gold:  {gold}')
    print(f'Pred:  {pred}')
    print()

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Input: be look for a temporary opportunity within a progressive and dynamic environment , please forward your cv today
Gold:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Pred:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Input: effective member of the operation leadership team customer interaction to develop positive relationship a
Gold:  ['O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'I-SOFT', 'I-SOFT', 'O', 'O', 'O', 'O', 'O']
Pred:  ['O', 'O', 'O', 'O', 'O', 'B-SOFT', 'I-SOFT', 'I-SOFT', 'I-SOFT', 'O', 'O', 'O', 'O', 'O']

Input: linux shell be able to demonstrate a systematic and analytical approach to
Gold:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O']
Pred:  ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O']



In [15]:
# Cell 15 — Full evaluation
MAX_EXAMPLES = 200

test_examples = list(ds['test'])
random.shuffle(test_examples)
test_examples = test_examples[:MAX_EXAMPLES]

true_tags = []
pred_tags = []

for example in tqdm(test_examples, desc='Evaluating'):
    tokens, gold = get_tokens_and_tags(example)
    pred = predict_with_rag_and_verify(tokens)
    true_tags.append(gold)
    pred_tags.append(pred)

print('\nDone.')

Evaluating: 100%|██████████| 200/200 [06:17<00:00,  1.89s/it]


Done.


In [16]:
# Cell 16 — Results
print('=== Sayfullina — Qwen2.5-3B Full SRICL (Phase 6) ===')
print(f'Examples evaluated: {len(true_tags)}')
print()
print(classification_report(true_tags, pred_tags))
f1 = f1_score(true_tags, pred_tags)
print(f'Strict F1: {f1:.4f}')
print()
print('Phase 1 (random 5-shot, no RAG):              0.1432 F1')
print('Phase 2 (RAG-1 + RAG-2, no verifier):         0.3154 F1')
print('Phase 3 (+ Verifier):                         0.3429 F1')
print('Phase 4 (SFT only, plain prompt):             0.8150 F1')
print('Phase 5 (SFT + ICL prompt, prompt mismatch):  0.7551 F1')
print('Phase 6 (SFT + ICL + RAG-1 + RAG-2 + Verify): <-- this run')
print('Paper SRICL target (Qwen2.5-14B full):        0.6118 F1')

=== Sayfullina — Qwen2.5-3B Full SRICL (Phase 6) ===
Examples evaluated: 200

              precision    recall  f1-score   support

        SOFT       0.78      0.78      0.78       201

   micro avg       0.78      0.78      0.78       201
   macro avg       0.78      0.78      0.78       201
weighted avg       0.78      0.78      0.78       201

Strict F1: 0.7800

Phase 1 (random 5-shot, no RAG):              0.1432 F1
Phase 2 (RAG-1 + RAG-2, no verifier):         0.3154 F1
Phase 3 (+ Verifier):                         0.3429 F1
Phase 4 (SFT only, plain prompt):             0.8150 F1
Phase 5 (SFT + ICL prompt, prompt mismatch):  0.7551 F1
Phase 6 (SFT + ICL + RAG-1 + RAG-2 + Verify): <-- this run
Paper SRICL target (Qwen2.5-14B full):        0.6118 F1
